# Exp07 · deteksi High dan evaluasi kelas Medium
Run all pada CPU. Simpan paket v3 dan kedua CSV v2 15 detik (all_data.csv dan good_acceptable.csv) versi repo di bawah Dataset. Eksperimen mengevaluasi LOPO dalam Train dan validation resmi.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/Penelitian-Adaptive-IDE')
DATASET_ROOT = PROJECT_ROOT / 'Dataset'
OUTPUT_ROOT = PROJECT_ROOT / 'models/batch-06/exp07/outputs'

## Validasi paket, fitur sensor dan metrik

In [ ]:
import hashlib
import json
import platform
import shutil
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import numpy as np
import pandas as pd
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import RobustScaler

HRV = ["rmssd_ms", "sdnn_ms"]
EYE = ["blink_rate_per_minute", "ear_left_mean", "ear_left_std", "ear_right_mean",
       "ear_right_std", "iris_size_estimate_mean", "iris_size_estimate_std", "iris_size_delta_mean"]
CORE = HRV + EYE
ID = ["participant_id", "episode_id"]
SEED = 42


def read_bundle(root, case):
    proxy = case == "v3"
    package = root / ("v3_behavioral_proxy" if proxy else "v4_self_reported_rating")
    target = "cognitive_load_label" if proxy else "difficulty_rating"
    labels = ["Low", "Medium", "High"] if proxy else list(range(1, 11))
    window_dir = Path("window_15s_step_5s")
    hashes = {}

    def read(relative, columns):
        path = package / relative
        hashes[str(Path(relative).as_posix())] = hashlib.sha256(path.read_bytes()).hexdigest()
        return pd.read_csv(path, usecols=columns)

    def attach(left, right, keys, validation):
        merged = left.merge(right, on=keys, how="left", validate=validation, indicator=True)
        if not merged.pop("_merge").eq("both").all():
            raise ValueError(f"CSV pendamping tidak cocok pada {keys}.")
        return merged

    keys = ["window_id", "participant_id", "session_id", target]
    data = read(window_dir / "good_acceptable.csv", keys + CORE)
    assignments = read(window_dir / "window_assignments.csv",
                       keys + ["episode_id"] + (["split"] if proxy else []))
    for table in (data, assignments):
        if table.window_id.duplicated().any() or table[keys].isna().any().any():
            raise ValueError("Kunci window tidak lengkap atau window_id berulang.")
    data = attach(data, assignments, keys, "one_to_one")
    evidence = read("episode_labels.csv" if proxy else "episode_ratings.csv",
                    ID + ["session_id", target])
    if evidence.episode_id.isna().any() or evidence.episode_id.duplicated().any():
        raise ValueError("episode_id pada bukti harus unik dan lengkap.")
    data = attach(data, evidence, ID + ["session_id", target], "many_to_one")
    if set(data[target]) != set(labels) or not np.isfinite(data[CORE].to_numpy(float)).all():
        raise ValueError("Periksa kelas target dan kelengkapan sepuluh fitur core.")
    if proxy:
        splits = read("participant_splits.csv", ["participant_id", "split"])
        if splits.participant_id.duplicated().any() or set(splits.split) != {"Train", "validation", "Test"}:
            raise ValueError("Periksa pembagian peserta v3.")
        data = attach(data, splits, ["participant_id", "split"], "many_to_one")
        weights = read(window_dir / "train_sample_weights.csv", ["window_id", *ID, target, "sample_weight"])
        train = data.split.eq("Train")
        if set(weights.window_id) != set(data.loc[train, "window_id"]) or weights.window_id.duplicated().any():
            raise ValueError("Daftar bobot berbeda dari window Train.")
        checked = attach(data.loc[train], weights, ["window_id", *ID, target], "one_to_one")
        if not np.isfinite(checked.sample_weight).all() or not checked.sample_weight.gt(0).all():
            raise ValueError("Bobot Train harus positif dan finite.")
        data["sample_weight"] = data.window_id.map(checked.set_index("window_id").sample_weight)
    return data.rename(columns={target: "actual"}), labels, hashes

def scores(table, labels):
    actual, predicted = table.actual, table.predicted
    row = dict(n=len(table), participants=table.participant_id.nunique(),
               accuracy=accuracy_score(actual, predicted),
               macro_f1=f1_score(actual, predicted, labels=labels, average="macro", zero_division=0))
    if len(labels) == 10:
        distance = (actual - predicted).abs()
        row.update(mae=distance.mean(), within_one=distance.le(1).mean())
    return row

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")

## Rekonstruksi referensi per fold dan model pembanding

In [ ]:
import itertools
from bisect import bisect_right
from collections import defaultdict
LEGACY_POLICY = 'behavioral_concordance_v1'
POLICY = 'behavioral_concordance_v2'
MIN_REFERENCE = 5

def make_reference(episodes, splits, *, policy=POLICY):
    """One first-exposure observation per person/item; only successful Train cases."""
    seen, items = set(), defaultdict(list)
    for row in sorted(episodes, key=lambda r: (r['start_ms'], r['episode_id'])):
        identity = row['participant_id'], row['item_key']
        if identity in seen:
            continue
        seen.add(identity)
        if (splits[row['participant_id']] == 'Train' and row['valid'] and row['completed']
                and (policy == LEGACY_POLICY or row.get('reference_eligible', True))
                and row['success_submission_type'] == 'manual' and row['active_seconds'] > 0):
            items[row['item_key']].append({'participant_id': row['participant_id'],
                'episode_id': row['episode_id'], 'active_seconds': row['active_seconds']})
    return {'policy': policy, 'minimum_other_people': MIN_REFERENCE, 'quantiles': [1/3,2/3],
            'quantile_method': 'linear', 'source_split': 'Train', 'self_reference': False,
            'items': {key: sorted(values, key=lambda v: v['participant_id']) for key, values in sorted(items.items())}}

def reference_for(reference, episode):
    values = [v for v in reference['items'].get(episode['item_key'], [])
              if v['participant_id'] != episode['participant_id']]
    result = {'participants': [v['participant_id'] for v in values], 'count': len(values),
                  'lower_seconds': None, 'upper_seconds': None, 'reason': ''}
    if len(values) < MIN_REFERENCE:
        result['reason'] = 'insufficient_reference'
    else:
        lower, upper = np.quantile([v['active_seconds'] for v in values], [1/3,2/3], method='linear')
        result.update(lower_seconds=float(lower), upper_seconds=float(upper))
        if not 0 < lower < upper:
            result['reason'] = 'degenerate_reference'
    return result

def label_episode(episode, reference):
    limits = reference_for(reference, episode)
    result = {'cognitive_load_label': '', 'label_reason': '', 'time_band': '', 'performance_band': '',
                  'reference_count': limits['count'], 'reference_participants': limits['participants'],
                  'lower_seconds': limits['lower_seconds'], 'upper_seconds': limits['upper_seconds']}
    if not episode['valid']:
        result['label_reason'] = 'invalid_event_trace'
    elif limits['reason']:
        result['label_reason'] = limits['reason']
    else:
        seconds, errors = episode['active_seconds'], episode['manual_errors']
        time_band = ('Low' if seconds <= limits['lower_seconds'] else
                     'Medium' if seconds <= limits['upper_seconds'] else 'High')
        performance = ('Low' if episode['completed'] and errors == 0 and episode['manual_submissions'] == 1
                       and episode['success_submission_type'] == 'manual' else
                       'Medium' if episode['completed'] and 1 <= errors <= 2 else 'High' if errors >= 3 else '')
        result.update(time_band=time_band, performance_band=performance)
        # An unfinished duration is only a lower bound: never use it for Low/Medium.
        if performance == time_band and (episode['completed'] or time_band == 'High'):
            result.update(cognitive_load_label=time_band, label_reason='concordant_behavioral_evidence')
        else:
            result['label_reason'] = 'conflicting_or_insufficient_behavior'
    return result

class WindowIndex:
    """Match only continuous active time from one episode, without rounding gaps."""

    def __init__(self, episodes):
        self.sessions = defaultdict(list)
        for episode in episodes:
            merged = []
            for start, end in sorted(episode['intervals']):
                if end <= start:
                    raise ValueError('Invalid active interval.')
                if merged and start <= merged[-1][1]:
                    merged[-1][1] = max(end, merged[-1][1])
                else:
                    merged.append([start,end])
            self.sessions[episode['session_id']].extend((start,end,episode) for start,end in merged)
        self.starts = {}
        for session, intervals in self.sessions.items():
            intervals.sort(key=lambda r: r[0])
            if any(b[0] < a[1] for a,b in itertools.pairwise(intervals)):
                raise ValueError('Overlapping active episodes.')
            self.starts[session] = [r[0] for r in intervals]

    def match(self, session_id, start, end):
        if end <= start:
            raise ValueError('Invalid window interval.')
        i = bisect_right(self.starts.get(session_id, []), start)-1
        if i >= 0:
            _, stop, episode = self.sessions[session_id][i]
            if end <= stop:
                return episode, ''
        return None, 'not_single_active_episode'

def epoch_ms(value):
    timestamp = pd.Timestamp(value)
    if pd.isna(timestamp) or timestamp.tzinfo is None:
        raise ValueError(f"Expected a timezone-aware timestamp: {value!r}")
    return timestamp.timestamp() * 1000

HEAD = ["head_pitch_mean", "head_pitch_std", "head_yaw_mean", "head_yaw_std"]
SENSORS = CORE + HEAD

def history_streams(path):
    keys = ["participant_id", "session_id"]
    bank = pd.read_csv(path, usecols=keys + ["window_start", "window_end"] + SENSORS)
    for key in ["window_start", "window_end"]:
        bank[key] = pd.to_datetime(bank[key], utc=True, format="mixed", errors="raise")
    if bank[keys + ["window_start", "window_end"]].isna().any().any() or not bank.window_end.gt(bank.window_start).all():
        raise ValueError("Interval sensor historis tidak valid.")
    bank = bank[np.isfinite(bank[SENSORS]).all(axis=1)]
    return dict(tuple(bank.sort_values("window_end", ascending=False).groupby(keys)))

def prior_reference(stream, start):
    # Newest first. Work only with disjoint windows that finished before this task.
    chosen = []
    limit = start
    if stream is not None:
        for index, left, right in stream[["window_start", "window_end"]].itertuples():
            if right <= limit:
                chosen.append(index)
                limit = left
                if len(chosen) == 12:
                    break
    if len(chosen) < 3:
        return pd.Series(np.nan, index=SENSORS), len(chosen), None
    return stream.loc[chosen, SENSORS].median(), len(chosen), stream.at[chosen[0], "window_end"]

"""Standalone notebook cells for participant-held-out evaluation of the v3 target."""

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

THREE = ["Low", "Medium", "High"]
TWO = ["Non-High", "High"]


def binary(y):
    return y.where(y.eq("High"), "Non-High")


def snapshot_hash(path, expected_lf_hash):
    data = path.read_bytes()
    if hashlib.sha256(data.replace(b"\r\n", b"\n")).hexdigest() != expected_lf_hash:
        raise ValueError(f"Gunakan snapshot v2 15 detik dari repo untuk exp07: {path.name}")
    return hashlib.sha256(data).hexdigest()


def build_pool(bank, all_windows, records, official):
    # Use all quality-eligible sensor windows, including episodes unlabeled by the full-Train reference.
    streams = history_streams(bank)
    frame = pd.read_csv(all_windows, usecols=["window_id", "session_id", "participant_id", "window_start",
                                             "window_end", "window_phase_mix", "quality_tier"] + SENSORS)
    frame = frame[frame.quality_tier.isin(["good", "acceptable"])]
    matcher = WindowIndex(records)
    matches = []
    for row in frame.itertuples():
        episode, _ = matcher.match(row.session_id, epoch_ms(row.window_start), epoch_ms(row.window_end))
        active = all(p["study_phase"] == "active_task" for p in json.loads(row.window_phase_mix)
                     if p["duration_seconds"] > 0)
        if episode and active:
            if episode["participant_id"] != row.participant_id:
                raise ValueError("Pemetaan peserta berbeda.")
            matches.append((row.Index, episode["episode_id"]))
    mapped = frame.loc[[i for i, _ in matches]].copy()
    mapped["episode_id"] = [e for _, e in matches]
    known = pd.DataFrame(records).set_index("episode_id")
    published = mapped[mapped.episode_id.map(known.cognitive_load_label).ne("")]
    cols = ["window_id", "participant_id", "episode_id"]
    pd.testing.assert_frame_equal(published[cols].sort_values(cols).reset_index(drop=True),
                                  official[cols].sort_values(cols).reset_index(drop=True), check_dtype=False)
    # No Test feature vectors enter model development.
    dev = mapped[mapped.episode_id.map(known.split).isin(["Train", "validation"])]
    if not np.isfinite(dev[SENSORS]).all().all():
        raise ValueError("Fitur sensor tidak lengkap.")
    x = dev.groupby("episode_id")[SENSORS].median()
    meta = known.loc[x.index, ["participant_id", "session_id", "start_ms", "split"]]
    refs = []
    for row in meta.itertuples():
        stream = streams.get((row.participant_id, row.session_id))
        start = pd.to_datetime(row.start_ms, unit="ms", utc=True)
        ref, _, _ = prior_reference(stream, start)
        refs.append(ref)
    return x, pd.DataFrame(refs, index=x.index), meta


def fold_labels(records, fit_people):
    roles = {row["participant_id"]: ("Train" if row["participant_id"] in fit_people else "held")
             for row in records}
    reference = make_reference(records, roles)
    return pd.Series({r["episode_id"]: label_episode(r, reference)["cognitive_load_label"]
                      for r in records})


def classify(x, y, people, held_index):
    persons = people.loc[y.index]
    weight = 1 / (persons.groupby(y).transform("nunique") * y.groupby([y, persons]).transform("size"))
    weight = weight / weight.mean()
    scaler = StandardScaler().fit(x.loc[y.index], sample_weight=weight)
    model = SVC(C=1, kernel="rbf", gamma="scale")
    model.fit(scaler.transform(x.loc[y.index]), y, sample_weight=weight)
    return pd.Series(model.predict(scaler.transform(x.loc[held_index])), index=held_index)


def run_fold(x, refs, meta, records, protocol, fold, fit_people, held_people, coverage):
    if fit_people & held_people:
        raise ValueError("Peserta muncul pada kedua sisi.")
    labels = fold_labels(records, fit_people).reindex(x.index)
    fit = labels.ne("") & meta.participant_id.isin(fit_people)
    held = labels.ne("") & meta.participant_id.isin(held_people)
    y, actual = labels[fit], labels[held]
    coverage.append(dict(protocol=protocol, fold=fold, fit_people=len(fit_people),
                         held_people=len(held_people), fit_episodes=len(y), held_episodes=len(actual),
                         held_sensor_episodes=int(meta.participant_id.isin(held_people).sum()),
                         fit_classes="|".join(sorted(y.unique()))))
    if actual.empty:
        return []
    if set(y.unique()) != set(THREE):
        raise ValueError(f"{fold}: kelas fitting tidak lengkap.")
    relative = x - refs.fillna(x.loc[y.index].median())
    people = meta.participant_id
    result = []

    def record(name, task, pred):
        rows = actual.to_frame("original").assign(participant_id=people, predicted=pred)
        rows["actual"] = actual if task == "three" else binary(actual)
        result.append(rows.assign(protocol=protocol, fold=fold, task=task, model=name).reset_index())

    record("majority", "three", y.value_counts().reindex(THREE).idxmax())
    two = binary(y)
    record("majority", "binary", two.value_counts().reindex(TWO).idxmax())
    gates = {}
    for name, view in [("eye", x[EYE]), ("relative_core_head", relative)]:
        pred = classify(view, y, people, actual.index)
        record(name, "three", pred)
        record(name + "_mapped", "binary", binary(pred))
        gates[name] = classify(view, two, people, actual.index)
        record(name + "_direct", "binary", gates[name])
    # Fixed cascade: eye gate for High, relative sensors discriminate Low/Medium.
    lower = classify(relative, y[y.ne("High")], people, actual.index)
    record("eye_gate_relative_low_medium", "three", lower.where(gates["eye"].ne("High"), "High"))
    return result


def run(dataset_root, output_root):
    official, _, hashes = read_bundle(dataset_root, "v3")
    ev = pd.read_csv(dataset_root / "v3_behavioral_proxy/episode_labels.csv",
                     keep_default_na=False, float_precision="round_trip")
    ev["intervals"] = ev.intervals.map(json.loads)
    records = ev.to_dict("records")
    people = pd.read_csv(dataset_root / "v3_behavioral_proxy/participant_splits.csv")
    train = set(people.loc[people.split.eq("Train"), "participant_id"])
    validation = set(people.loc[people.split.eq("validation"), "participant_id"])
    relabeled = fold_labels(records, train)
    if not ev.episode_id.map(relabeled).eq(ev.cognitive_load_label).all():
        raise ValueError("Rekonstruksi label full Train berbeda dari paket resmi.")
    bank = dataset_root / "v2_session_phase/window_15s_step_5s/good_acceptable.csv"
    digest = snapshot_hash(bank, "39e72fd8504bb57d83e2edd91a371e904ba6203ad5a3df90fa560a3eaa715722")
    all_windows = bank.with_name("all_data.csv")
    all_digest = snapshot_hash(all_windows, "acbef9a60f4114de460ee3c57ca94ff5cae0d48b67229d8b14283d173fef1355")
    x, refs, meta = build_pool(bank, all_windows, records, official)
    coverage, parts = [], []
    for person in sorted(train):
        parts.extend(run_fold(x, refs, meta, records, "Train_LOPO", person,
                              train - {person}, {person}, coverage))
    parts.extend(run_fold(x, refs, meta, records, "validation", "validation", train, validation, coverage))
    predictions = pd.concat(parts, ignore_index=True)
    results, confusions = [], []
    for (protocol, task, name), group in predictions.groupby(["protocol", "task", "model"]):
        labels = THREE if task == "three" else TWO
        assert group.episode_id.is_unique
        cm = confusion_matrix(group.actual, group.predicted, labels=labels)
        info = dict(protocol=protocol, task=task, model=name)
        results.append({**info, "episodes": len(group), "participants": group.participant_id.nunique(),
                        "correct": int(cm.trace()), "accuracy": cm.trace()/len(group),
                        "macro_f1": f1_score(group.actual, group.predicted, labels=labels, average="macro", zero_division=0),
                        "high_tp": int(cm[-1,-1]), "high_fn": int(cm[-1,:-1].sum()),
                        "high_fp": int(cm[:-1,-1].sum())})
        for original, rows in group.groupby("original"):
            confusions.append({**info, "original": original, **rows.predicted.value_counts().to_dict()})
    out = output_root / ("v3_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8])
    out.mkdir(parents=True, exist_ok=False)
    predictions.to_csv(out / "episode_predictions.csv", index=False)
    pd.DataFrame(coverage).to_csv(out / "fold_coverage.csv", index=False)
    pd.DataFrame(confusions).fillna(0).to_csv(out / "confusion_by_original_label.csv", index=False)
    summary = pd.DataFrame(results)
    summary.to_csv(out / "metrics.csv", index=False)
    distribution = []
    for unit, frame in [("window", official), ("episode", official.drop_duplicates("episode_id"))]:
        distribution.append(frame.groupby(["split", "actual"]).size().rename("count").reset_index().assign(unit=unit))
    pd.concat(distribution).to_csv(out / "distribution.csv", index=False)
    save_json(out / "manifest.json", {"v3": hashes, "v2_history_sha256": digest, "v2_all_sha256": all_digest, "sklearn": sklearn.__version__,
              "pandas": pd.__version__, "numpy": np.__version__, "target": {"Low": "Non-High", "Medium": "Non-High", "High": "High"}})
    print(summary.to_string(index=False))
    archive = shutil.make_archive(str(out), "zip", root_dir=out.parent, base_dir=out.name)
    print("ZIP:", archive)
    return Path(archive)


## Jalankan v3 dan simpan hasil

In [ ]:
archive = run(DATASET_ROOT, OUTPUT_ROOT)